# Lecture: DQN for the LunarLander environment

LunarLander-v3 is a significantly harder control problem than CartPole.
A spacecraft must land smoothly on a landing pad using discrete thruster commands.

## Environment Overview

- **Goal:** Land the craft on the pad between the flags without crashing.
- **Observation space:** 8 continuous values
  - x/y position, x/y velocity, angle, angular velocity, left/right leg contact (binary)
- **Action space:** Discrete with 4 actions
  - 0: Do nothing, 1: Fire left engine, 2: Fire main engine, 3: Fire right engine
- **Reward:**
  - +100–140 for landing on the pad, −100 for crashing
  - +10 per leg in contact with the ground
  - −0.3 per main-engine firing (fuel cost)
- **Solved:** Average score ≥ 200 over 100 consecutive episodes

> **Training time:** Expect ~15–25 min for DQN and ~10–15 min for NaiveDQN on Colab (CPU-bound Box2D simulation).

Run the following cell only if you are working with Google Colab to copy the required .py files into the root directory. If you are working locally just ignore this cell!

In [ ]:
!git clone https://github.com/Fjoelsak/RL.git
!cp RL/06_Value_Function_Approximation/DQN_agent.py ./
!cp RL/06_Value_Function_Approximation/plot_utils.py ./
!pip install -q gymnasium[box2d]

### Get to know the environment

Run a few random steps to observe the state and reward structure.

In [ ]:
import gymnasium as gym

env = gym.make("LunarLander-v3")
obs, _ = env.reset()

print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print("Sample obs:       ", obs)

total_reward = 0
for _ in range(200):
    obs, reward, terminated, truncated, _ = env.step(env.action_space.sample())
    total_reward += reward
    if terminated or truncated:
        break
print("Random-policy episode reward:", round(total_reward, 2))
env.close()

## DQN Training — Multi-Seed Stability Analysis

We run DQN with `N_SEEDS` independent seeds to assess training stability.
Each seed uses a fresh environment and agent with identical hyperparameters;
only the log directory differs.
The shaded band in the plot shows ±1 standard deviation across seeds.

We use a larger network (128 → 128) compared to CartPole (24 → 48) because
LunarLander has an 8-dimensional state space and a more complex reward landscape.

> **Training time on Colab:** ~15–25 min per seed (CPU-bound Box2D simulation).

In [ ]:
from DQN_agent import DQNAgent
import gymnasium as gym

N_SEEDS = 3

base_config = {
    "EPISODES":              1_000,
    "REPLAY_MEMORY_SIZE":    100_000,
    "MINIMUM_REPLAY_MEMORY": 1_000,
    "TRAIN_FREQUENCY":       4,
    "MINIBATCH_SIZE":        64,
    "UPDATE_TARGETNW_STEPS": 1_000,
    "LEARNING_RATE":         0.001,
    "EPSILON":               1,
    "EPSILON_DECAY":         0.995,
    "MINIMUM_EPSILON":       0.01,
    "DISCOUNT":              0.99,
    "HIDDEN_DIMS":           (128, 128),
    "VISUALIZATION":         False,
}

In [ ]:
import numpy as np

agents         = []
all_avg_scores = []

for seed in range(N_SEEDS):
    print(f"\n=== Seed {seed + 1}/{N_SEEDS} ===")
    config = {**base_config, 'LOGS': f'logs/DQN_LunarLander_seed{seed}/'}
    env    = gym.make("LunarLander-v3")
    agent  = DQNAgent(env, config)
    agent.train(env)
    agents.append(agent)
    all_avg_scores.append(agent.average_score_100_episodes)
    env.close()

agent = agents[0]  # keep reference for video rendering below

In [ ]:
import matplotlib.pyplot as plt

data     = np.array(all_avg_scores)  # shape (N_SEEDS, episodes)
mean     = data.mean(axis=0)
std      = data.std(axis=0)
episodes = np.arange(len(mean))

fig, ax = plt.subplots(figsize=(10, 5))
for scores in all_avg_scores:
    ax.plot(episodes, scores, alpha=0.25, linewidth=1, color="steelblue")
ax.plot(episodes, mean, color="steelblue", linewidth=2.5,
        label=f"Mean ({N_SEEDS} seeds)")
ax.fill_between(episodes, mean - std, mean + std,
                alpha=0.2, color="steelblue", label="±1 std")
ax.axhline(200, color="gray", linestyle="--", linewidth=1,
           label="Solved threshold (200)")
ax.set_xlabel("Episode")
ax.set_ylabel("Average reward (100 episodes)")
ax.set_title("DQN on LunarLander — training stability across seeds")
ax.legend()
plt.tight_layout()
plt.show()

### Note: Use the following code if you are using Google Colab

In [ ]:
import gymnasium as gym
import imageio
from IPython.display import HTML
from base64 import b64encode

render_env = gym.make("LunarLander-v3", render_mode="rgb_array")
cur_state, _ = render_env.reset()

frames = []
done = False
while not done:
    frames.append(render_env.render())
    action = agent._predict_action(cur_state)
    cur_state, _, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
frames.append(render_env.render())
render_env.close()

video_path = "./lunarlander_dqn.mp4"
imageio.mimsave(video_path, frames, fps=30)

In [ ]:
mp4 = open(video_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=500 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")